In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score, roc_curve, precision_recall_curve
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE
import json
from sklearn.svm import LinearSVC

In [ ]:
# 加载数据集
data_path = '/data/jxiao/gas_data/gas_new/values_refine/RGB_diff_refine_label_new.csv'

diff = pd.read_csv(data_path, index_col='id')
diff_n2 = (np.sqrt(diff.iloc[:, :-1]**2)).T.groupby(np.arange(diff.shape[1]-1) // 3).sum().T

X = diff.drop('label', axis=1, inplace=False).to_numpy()
X_n2 = diff_n2.to_numpy()
y = diff['label'].to_numpy().astype(int)
len(y), sum(y)

In [75]:
N_SPLITS = 5
use_smote = False

In [ ]:
skf = StratifiedKFold(n_splits=N_SPLITS, 
                      shuffle=True, 
                      random_state=42
                    )
all_acc, all_f1, all_auc, all_precision, all_recall = [], [], [], [], [] # eval metrics
y_pred = []
p_pred = []
all_fpr = []
all_tpr = []
for fold, (train_idx, val_idx) in enumerate(skf.split(X_n2, y)):
    print(f"===== Fold {fold + 1} =====")
    X_train, X_val = X_n2[train_idx], X_n2[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    # --- 标准化 ---
    scaler = StandardScaler()
    scaler.fit(X_train)

    X_train = scaler.transform(X_train)
    X_val = scaler.transform(X_val)

    # --- SMOTE 过采样 ---
    if use_smote:
        smote = SMOTE(random_state=42)
        X_train, y_train = smote.fit_resample(X_train, y_train)

    model = LogisticRegression(
        penalty="l2",        # L2 正则（默认，适合小样本）
        C=.8,               # 正则强度（越小正则越强）
        solver="liblinear",
        max_iter=1000,
        class_weight=None   # 若类别不平衡可用 "balanced"
        )
    
    model.fit(X_train, y_train)
    y_pred.append(model.predict(X_val))
    p_pred.append(model.predict_proba(X_val))

    acc = accuracy_score(y_val, y_pred[-1])
    f1 = f1_score(y_val, y_pred[-1])
    auc = roc_auc_score(y_val, p_pred[-1][:, 1])
    precision = precision_score(y_val, y_pred[-1])
    recall = recall_score(y_val, y_pred[-1])
    fpr, tpr, _ = roc_curve(y_val, p_pred[-1][:, 1])

    all_fpr.append(fpr)
    all_tpr.append(tpr)
    all_acc.append(acc)
    all_f1.append(f1)
    all_auc.append(auc)
    all_precision.append(precision)
    all_recall.append(recall)


In [ ]:
print(f"===== Overall Results ({N_SPLITS}-Fold Avg) =====")
print(f"ACC={np.mean(all_acc):.3f} ± {np.std(all_acc):.3f}")
print(f"F1 ={np.mean(all_f1):.3f} ± {np.std(all_f1):.3f}")
print(f"AUC={np.mean(all_auc):.3f} ± {np.std(all_auc):.3f}")
print(f"Precision={np.mean(all_precision):.3f} ± {np.std(all_precision):.3f}")
print(f"Recall={np.mean(all_recall):.3f} ± {np.std(all_recall):.3f}")

In [ ]:
fig, ax = plt.subplots(1, 5, figsize=(20, 4), sharey=True)

for i in range(5):
    ax[i].plot(all_fpr[i], all_tpr[i], color='darkorange', lw=2, label=f'ROC curve (AUC = {all_auc[i]:.3f})')
    ax[i].plot([0, 1], [0, 1], color='navy', lw=1.5, linestyle='--')
    ax[i].set_title(f'Fold {i+1}', fontsize=14)
    ax[i].set_xlabel('False Positive Rate', fontsize=14)
    ax[i].legend(loc='lower right')
    # ax[i].grid(True, linestyle='--', alpha=0.5)

# plt.legend(loc='lower right')
fig.suptitle('ROC Curve', fontsize=14, fontweight='bold')
# fig.supxlabel('False positive rate', fontsize=14)
fig.supylabel('True positive rate', fontsize=14)
plt.tight_layout(rect=[0.005, -0.05, 1, 0.999])
plt.show()

In [6]:
js_val = {}
js_val['threshold'] = 0.5
js_val['accuracy'] = all_acc
js_val['f1_score'] = all_f1
js_val['auc'] = all_auc
js_val['precision'] = all_precision
js_val['recall'] = all_recall
js_val['fpr'] = [fpr.tolist() for fpr in all_fpr]
js_val['tpr'] = [tpr.tolist() for tpr in all_tpr]

with open('weights/RGB_diff_refine_label_new_n2/LR_val.json', 'w') as f:
    json.dump(js_val, f)

In [ ]:
skf = StratifiedKFold(n_splits=N_SPLITS, 
                      shuffle=True, 
                      random_state=42
                    )
all_acc, all_f1, all_auc, all_precision, all_recall = [], [], [], [], [] # eval metrics
y_pred = []
score = []
all_fpr = []
all_tpr = []
for fold, (train_idx, val_idx) in enumerate(skf.split(X_n2, y)):
    print(f"===== Fold {fold + 1} =====")
    X_train, X_val = X_n2[train_idx], X_n2[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    # --- 标准化 ---
    scaler = StandardScaler()
    scaler.fit(X_train)

    X_train = scaler.transform(X_train)
    X_val = scaler.transform(X_val)

    # --- SMOTE 过采样 ---
    if use_smote:
        smote = SMOTE(random_state=42)
        X_train, y_train = smote.fit_resample(X_train, y_train)

    ## paper params
    model = LinearSVC(
            C=.1,               # 正则强度（越小越强）
            loss="squared_hinge",
            max_iter=3000,
            random_state=None
            )
    
    model.fit(X_train, y_train)
    y_pred.append(model.predict(X_val))
    score.append(model.decision_function(X_val))

    acc = accuracy_score(y_val, y_pred[-1])
    f1 = f1_score(y_val, y_pred[-1])
    auc = roc_auc_score(y_val, score[-1])
    precision = precision_score(y_val, y_pred[-1])
    recall = recall_score(y_val, y_pred[-1])
    fpr, tpr, thresholds = roc_curve(y_val, score[-1])

    all_acc.append(acc)
    all_f1.append(f1)
    all_auc.append(auc)
    all_precision.append(precision)
    all_recall.append(recall)
    all_fpr.append(fpr)
    all_tpr.append(tpr)

In [ ]:
print(f"===== Overall Results ({N_SPLITS}-Fold Avg) =====")
print(f"ACC={np.mean(all_acc):.3f} ± {np.std(all_acc):.3f}")
print(f"F1 ={np.mean(all_f1):.3f} ± {np.std(all_f1):.3f}")
print(f"AUC={np.mean(all_auc):.3f} ± {np.std(all_auc):.3f}")
print(f"Precision={np.mean(all_precision):.3f} ± {np.std(all_precision):.3f}")
print(f"Recall={np.mean(all_recall):.3f} ± {np.std(all_recall):.3f}")

In [30]:
js_val = {}
js_val['threshold'] = 0.5
js_val['accuracy'] = all_acc
js_val['f1_score'] = all_f1
js_val['auc'] = all_auc
js_val['precision'] = all_precision
js_val['recall'] = all_recall
js_val['fpr'] = [fpr.tolist() for fpr in all_fpr]
js_val['tpr'] = [tpr.tolist() for tpr in all_tpr]

with open('weights/RGB_diff_refine_label_new_n2/SVM_val.json', 'w') as f:
    json.dump(js_val, f)

In [ ]:
fig, ax = plt.subplots(1, 5, figsize=(20, 4), sharey=True)

for i in range(5):
    ax[i].plot(all_fpr[i], all_tpr[i], color='darkorange', lw=2, label=f'ROC curve (AUC = {all_auc[i]:.3f})')
    ax[i].plot([0, 1], [0, 1], color='navy', lw=1.5, linestyle='--')
    ax[i].set_title(f'Fold {i+1}', fontsize=14)
    ax[i].set_xlabel('False Positive Rate', fontsize=14)
    ax[i].legend(loc='lower right')
    # ax[i].grid(True, linestyle='--', alpha=0.5)

# plt.legend(loc='lower right')
fig.suptitle('ROC Curve', fontsize=14, fontweight='bold')
# fig.supxlabel('False positive rate', fontsize=14)
fig.supylabel('True positive rate', fontsize=14)
plt.tight_layout(rect=[0.005, -0.05, 1, 0.999])
plt.show()